# Trace an agent in Sentry, from the Messages API to Managed Agents

You can build the same agent on the Messages API, the Agent SDK, or Claude Managed Agents, and each one can send OpenTelemetry (OTel) spans to the same Sentry project. What differs is who emits the spans and how you turn them on: a tracer provider in your code, environment variables, or a destination in the Claude Console. The Claude API emits no span for a tool your own code runs, so that one is yours to emit. This notebook builds one site-health agent five ways and traces each, and adds the environment variables for Claude Code. All the tracing is in beta, and so are message threads and the tool runner, so span names and options can change between versions.

A *span* is one timed step with attributes. A *trace* is the tree of spans from one piece of work. *OTLP* is the protocol that carries them, and Sentry accepts it directly, so you don't need a Sentry SDK. A *tracer provider* is the OTel object that creates spans in your process and passes them to *exporters*, which send them somewhere.

| | Who runs the loop | Who emits spans | Turn tracing on with |
|---|---|---|---|
| Manual loop (section 2) | Your code | Anthropic SDK | A tracer provider |
| Message thread (section 3) | Your code | Anthropic SDK | A tracer provider |
| Tool runner (section 4) | Anthropic SDK | Anthropic SDK | A tracer provider |
| Claude Code (section 5) | Claude Code | Claude Code | Environment variables |
| Agent SDK (section 6) | Claude Code, as a child process | Claude Code | The same variables, through `env` |
| Managed Agents (section 8) | Anthropic's servers | Claude API, plus the Anthropic SDK for your tools | A Console destination (section 7), plus a tracer provider |

The Claude API and the Anthropic SDKs export traces with span names that start with `anthropic.`. Claude Code and the Agent SDK export metrics, log events, and traces with names that start with `claude_code.`.

For a span-by-span, attribute-by-attribute reference, see [Trace Claude API requests and Managed Agents sessions with OpenTelemetry](opentelemetry_tracing.ipynb).

## 1. Set up

**Prerequisites**

- Python 3.11 or later
- `ANTHROPIC_API_KEY`, in the environment or in a `.env` file. Section 8 needs access to Managed Agents, so skip sections 8 and 10 if you don't have it
- Optional: `SENTRY_DSN`, from **Project settings > Client Keys (DSN)** in Sentry. Without it, spans stay in memory and this notebook prints them
- Optional, for section 7: the admin role in your Claude Console organization

The `anthropic` package depends on the OTel API only, which records nothing by itself. Install the OTel SDK and the OTLP exporter next to it. `claude-agent-sdk` includes Claude Code. Pin the versions in your own app.

In [1]:
%%capture
%pip install -q "anthropic>=2.9.0,<3" "claude-agent-sdk>=0.2.183" opentelemetry-sdk opentelemetry-exporter-otlp-proto-http python-dotenv

A Sentry DSN looks like `https://<key>@<host>/<project id>`. Sentry's OTLP endpoint and auth header come from those three parts. The next cell turns a DSN into the two standard OTel environment variables, which the exporters in this notebook read. The header is named `x-sentry-auth` and its value is `sentry sentry_key=<key>`. If you change the DSN later, restart the kernel, because a process can register one tracer provider only.

Register the tracer provider before the first request. The Anthropic SDK picks up the global provider, so the client needs no tracing arguments. By default its spans carry metadata only: model, tokens, timing, and status, with no prompts or responses.

In [2]:
import json
import os
import time
from urllib.parse import urlsplit

from anthropic import Anthropic, DefaultHttpxClient
from dotenv import load_dotenv
from opentelemetry import trace
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor, SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

load_dotenv()

MODEL = os.environ.get("COOKBOOK_MODEL", "claude-haiku-5-5")
SENTRY_DSN = os.environ.get("SENTRY_DSN")

if SENTRY_DSN:
    dsn = urlsplit(SENTRY_DSN)
    host = dsn.netloc.rsplit("@", 1)[-1]
    prefix, project_id = dsn.path.rstrip("/").rsplit("/", 1)
    for name in ("ENDPOINT", "HEADERS"):  # per-signal variables would win over the ones below
        os.environ.pop(f"OTEL_EXPORTER_OTLP_TRACES_{name}", None)
        os.environ.pop(f"OTEL_EXPORTER_OTLP_LOGS_{name}", None)
    os.environ["OTEL_EXPORTER_OTLP_ENDPOINT"] = (
        f"{dsn.scheme}://{host}{prefix}/api/{project_id}/integration/otlp"
    )
    os.environ["OTEL_EXPORTER_OTLP_HEADERS"] = f"x-sentry-auth=sentry sentry_key={dsn.username}"

provider = TracerProvider(resource=Resource.create({"service.name": "site-health"}))
memory = InMemorySpanExporter()  # a local copy, so this notebook can print spans
provider.add_span_processor(SimpleSpanProcessor(memory))
if SENTRY_DSN:
    provider.add_span_processor(BatchSpanProcessor(OTLPSpanExporter()))
trace.set_tracer_provider(provider)
tracer = trace.get_tracer("site-health")

print("Exporting to Sentry" if SENTRY_DSN else "SENTRY_DSN is not set, so spans stay in memory")

SENTRY_DSN is not set, so spans stay in memory


The agent has one tool, `check_pages`, which requests four pages one after another. Each probe opens its own span, so the trace shows the time inside the tool. `probe` sleeps for one second in place of a real HTTP request.

`log_request` prints the size of each Messages API request, so you can compare the first three versions. `print_spans` prints the spans collected since the last call as a tree. The first column is when each span started.

In [3]:
PAGES = ["/", "/pricing", "/docs", "/login"]
QUESTION = "Is my site healthy? Check the pages, then answer in one sentence."

CHECK_PAGES = {
    "name": "check_pages",
    "description": "Check that every page on the website responds.",
    "input_schema": {"type": "object", "properties": {}},
}


def probe(page: str) -> dict:
    with tracer.start_as_current_span(f"probe {page}"):
        time.sleep(1)  # stands in for an HTTP request
        return {"page": page, "status": 200}


def check_pages() -> str:
    return json.dumps([probe(page) for page in PAGES])


def log_request(request) -> None:
    if request.url.path == "/v1/messages":
        body = json.loads(request.content)
        print(
            f"request: {len(body['messages'])} message(s), "
            f"tools {'sent' if body.get('tools') else 'not sent'}, {len(request.content)} bytes"
        )


def print_spans(extra=()) -> None:
    # extra: (id, parent id, name, start, end, note) tuples from outside this process
    rows = list(extra)
    for s in memory.get_finished_spans():
        usage = s.attributes.get("gen_ai.usage.input_tokens")
        out = s.attributes.get("gen_ai.usage.output_tokens")
        note = f"{usage} in, {out} out" if usage else ""
        parent = s.parent.span_id if s.parent else None
        rows.append((s.context.span_id, parent, s.name, s.start_time, s.end_time, note))
    memory.clear()
    rows.sort(key=lambda row: row[3])
    ids = {row[0] for row in rows}

    def walk(parent_id, depth):
        for span_id, parent, name, start, end, note in rows:
            if (parent if parent in ids else None) == parent_id:
                offset, seconds = (start - rows[0][3]) / 1e9, (end - start) / 1e9
                print(f"{offset:5.1f}s  {'    ' * depth}{name}  {seconds:.1f}s  {note}".rstrip())
                walk(span_id, depth + 1)

    walk(None, 0)


def final_text(message) -> str:
    return "".join(block.text for block in message.content if block.type == "text")


client = Anthropic(http_client=DefaultHttpxClient(event_hooks={"request": [log_request]}))

## 2. Manual loop on the Messages API

Your code keeps the conversation and sends all of it on every request, with the tool definitions. The SDK emits one span per request. Your code runs the tool, so the SDK doesn't know about it: open a span yourself, or the tool's time shows up as a gap.

Wrap the loop in one root span. Without it, each request starts its own trace.

In [4]:
messages = [{"role": "user", "content": QUESTION}]

with tracer.start_as_current_span("health_check manual loop"):
    while True:
        response = client.messages.create(
            model=MODEL,
            max_tokens=16000,
            tools=[CHECK_PAGES],
            messages=messages,  # everything, every time
        )
        messages.append({"role": "assistant", "content": response.content})
        if response.stop_reason != "tool_use":
            break

        results = []
        for block in response.content:
            if block.type == "tool_use":
                with tracer.start_as_current_span(f"run_tool {block.name}"):
                    output = check_pages(**block.input)
                results.append({"type": "tool_result", "tool_use_id": block.id, "content": output})
        messages.append({"role": "user", "content": results})

print(final_text(response), "\n")
print_spans()

request: 1 message(s), tools sent, 299 bytes


request: 3 message(s), tools sent, 720 bytes


Yes, your site looks healthy: all four pages checked (/, /pricing, /docs, and /login) returned a 200 OK status. 

  0.0s  health_check manual loop  6.1s
  0.0s      anthropic.messages.create  1.5s  371 in, 27 out
  1.5s      run_tool check_pages  4.0s
  1.5s          probe /  1.0s
  2.5s          probe /pricing  1.0s
  3.5s          probe /docs  1.0s
  4.5s          probe /login  1.0s
  5.5s      anthropic.messages.create  0.6s  466 in, 46 out


## 3. Message thread

With a [message thread](https://platform.claude.com/docs/en/build-with-claude/message-threads), the API holds the conversation between turns. The first `send` creates the thread with the tools. Later sends carry the new messages only. The helper keeps the conversation in memory and replays it if the thread has expired, which happens five minutes after its last request.

The spans match section 2. Compare the request sizes.

In [5]:
thread = client.beta.messages.thread(model=MODEL, max_tokens=16000, tools=[CHECK_PAGES])

with tracer.start_as_current_span("health_check thread"):
    response = thread.send({"role": "user", "content": QUESTION})

    while response.stop_reason == "tool_use":
        results = []
        for block in response.content:
            if block.type == "tool_use":
                with tracer.start_as_current_span(f"run_tool {block.name}"):
                    output = check_pages(**block.input)
                results.append({"type": "tool_result", "tool_use_id": block.id, "content": output})

        # Later sends: only the new message goes over the wire
        response = thread.send({"role": "user", "content": results})

print(final_text(response), "\n")
print_spans()

request: 1 message(s), tools sent, 326 bytes


request: 1 message(s), tools not sent, 413 bytes


Yes, all four pages checked (/, /pricing, /docs, and /login) returned a 200 OK status, so your site looks healthy. 

  0.0s  health_check thread  5.9s
  0.0s      anthropic.messages.create  1.3s  371 in, 27 out
  1.3s      run_tool check_pages  4.0s
  1.3s          probe /  1.0s
  2.3s          probe /pricing  1.0s
  3.3s          probe /docs  1.0s
  4.3s          probe /login  1.0s
  5.3s      anthropic.messages.create  0.6s  466 in, 47 out


## 4. Tool runner

The [tool runner](https://platform.claude.com/docs/en/agents-and-tools/tool-use/tool-runner) writes the loop for you and uses a message thread by default. Because the SDK now runs the tool, it emits the tool span too: `anthropic.custom_tool_use check_pages`. Spans your tool opens nest under it.

In [6]:
from anthropic import beta_tool


@beta_tool(name="check_pages")
def check_pages_tool() -> str:
    """Check that every page on the website responds."""
    return check_pages()


with tracer.start_as_current_span("health_check tool runner"):
    runner = client.beta.messages.tool_runner(
        model=MODEL,
        max_tokens=16000,
        tools=[check_pages_tool],
        messages=[{"role": "user", "content": QUESTION}],
    )
    message = runner.until_done()

print(final_text(message), "\n")
print_spans()

request: 1 message(s), tools sent, 355 bytes


request: 1 message(s), tools not sent, 413 bytes


Yes, your site looks healthy: all four pages I checked (/, /pricing, /docs, and /login) returned a 200 OK status. 

  0.0s  health_check tool runner  6.0s
  0.0s      anthropic.messages.tool_runner  6.0s
  0.0s          anthropic.messages.create  1.4s  381 in, 27 out
  1.4s          anthropic.custom_tool_use check_pages  4.0s
  1.4s              probe /  1.0s
  2.4s              probe /pricing  1.0s
  3.4s              probe /docs  1.0s
  4.4s              probe /login  1.0s
  5.4s          anthropic.messages.create  0.6s  476 in, 47 out


### Record prompts and responses

To add content to the SDK's spans, create the client with `open_telemetry={"traces": {"content_mode": "content"}}`, or set `ANTHROPIC_OPEN_TELEMETRY_TRACES_CONTENT_MODE=content`. Prompts often hold user data, and Sentry keeps what it receives under your project's retention and scrubbing settings. Keep the default, `metadata_only`, until your data policy covers that.

## 5. Claude Code

Claude Code has its own OTel export, configured with environment variables. Traces are in beta and need a second flag. Run this in a shell, not in this notebook:

```bash
export CLAUDE_CODE_ENABLE_TELEMETRY=1
export CLAUDE_CODE_ENHANCED_TELEMETRY_BETA=1  # for traces

export OTEL_TRACES_EXPORTER=otlp
export OTEL_LOGS_EXPORTER=otlp
export OTEL_EXPORTER_OTLP_PROTOCOL=http/protobuf

# From your Sentry DSN: https://<key>@<host>/<project id>
export OTEL_EXPORTER_OTLP_ENDPOINT="https://<host>/api/<project id>/integration/otlp"
export OTEL_EXPORTER_OTLP_HEADERS="x-sentry-auth=sentry sentry_key=<key>"

claude
```

`OTEL_METRICS_EXPORTER` is left out because [Sentry doesn't accept OTLP metrics](https://docs.sentry.io/concepts/otlp/). The [monitoring guide](https://code.claude.com/docs/en/monitoring-usage) lists every variable. Prompts and tool content are off by default, and its `OTEL_LOG_*` settings turn them on. One of them, `OTEL_LOG_TOOL_CONTENT`, writes span events, which Sentry drops. To tell your agents apart in Sentry, set `OTEL_SERVICE_NAME`, which defaults to `claude-code`.

## 6. Agent SDK

The Agent SDK runs Claude Code as a child process, and that process exports the spans. Pass it the variables from section 5 through `ClaudeAgentOptions(env=...)`. In Python, `env` adds to the inherited environment. In TypeScript, `env` replaces it, so spread `...process.env` into it.

Call `query()` inside an active span. The SDK hands the trace context to the child process, so Claude Code's spans join your trace.

The child's spans never reach this notebook's in-memory exporter. If you didn't set `SENTRY_DSN`, the next cell starts a small OTLP receiver on localhost to stand in for Sentry.

In [7]:
import gzip
import threading
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

from opentelemetry.proto.collector.trace.v1.trace_service_pb2 import ExportTraceServiceRequest

received = []  # spans Claude Code sent to the local receiver


class Receiver(BaseHTTPRequestHandler):
    def do_POST(self):
        body = self.rfile.read(int(self.headers.get("content-length", 0)))
        if self.headers.get("content-encoding") == "gzip":
            body = gzip.decompress(body)
        if self.path.endswith("/v1/traces"):
            request = ExportTraceServiceRequest.FromString(body)
            for resource in request.resource_spans:
                for scope in resource.scope_spans:
                    received.extend(scope.spans)
        self.send_response(200)
        self.end_headers()

    def log_message(self, *args):
        pass


if SENTRY_DSN:
    endpoint = os.environ["OTEL_EXPORTER_OTLP_ENDPOINT"]
    headers = os.environ["OTEL_EXPORTER_OTLP_HEADERS"]
else:
    receiver = ThreadingHTTPServer(("127.0.0.1", 0), Receiver)
    threading.Thread(target=receiver.serve_forever, daemon=True).start()
    endpoint, headers = f"http://127.0.0.1:{receiver.server_port}", ""

OTEL = {
    "CLAUDE_CODE_ENABLE_TELEMETRY": "1",
    "CLAUDE_CODE_ENHANCED_TELEMETRY_BETA": "1",  # for traces
    "OTEL_SERVICE_NAME": "site-health",
    "OTEL_TRACES_EXPORTER": "otlp",
    "OTEL_LOGS_EXPORTER": "otlp",
    "OTEL_EXPORTER_OTLP_PROTOCOL": "http/protobuf",
    "OTEL_EXPORTER_OTLP_ENDPOINT": endpoint,
    "OTEL_EXPORTER_OTLP_HEADERS": headers,
}

In [8]:
import asyncio

from claude_agent_sdk import (
    ClaudeAgentOptions,
    ResultMessage,
    create_sdk_mcp_server,
    query,
    tool,
)


async def probe_async(page: str) -> dict:
    with tracer.start_as_current_span(f"probe {page}"):
        await asyncio.sleep(1)  # stands in for an HTTP request
        return {"page": page, "status": 200}


@tool("check_pages", "Check that every page on the website responds.", {})
async def check_pages_mcp(args):
    results = [await probe_async(page) for page in PAGES]
    return {"content": [{"type": "text", "text": json.dumps(results)}]}


options = ClaudeAgentOptions(
    model=MODEL,
    mcp_servers={"site": create_sdk_mcp_server(name="site", tools=[check_pages_mcp])},
    allowed_tools=["mcp__site__check_pages"],
    setting_sources=[],  # ignore this machine's Claude Code settings
    env=OTEL,
)

with tracer.start_as_current_span("health_check agent sdk"):
    async for message in query(prompt=QUESTION, options=options):
        if isinstance(message, ResultMessage):
            print(message.result, "\n")

claude_code_spans = []
for span in received:
    attributes = {a.key: a.value.string_value for a in span.attributes}
    claude_code_spans.append(
        (
            int.from_bytes(span.span_id, "big"),
            int.from_bytes(span.parent_span_id, "big"),
            span.name,
            span.start_time_unix_nano,
            span.end_time_unix_nano,
            attributes.get("tool_name", ""),
        )
    )
print_spans(claude_code_spans)

Yes, your site looks healthy: all four pages I checked (/, /pricing, /docs, /login) returned HTTP 200. 



  0.0s  health_check agent sdk  9.6s
  0.5s      initialize  0.0s
  0.6s      notifications/initialized  0.0s
  0.6s      tools/list  0.0s
  0.6s      claude_code.interaction  8.8s
  1.1s          claude_code.llm_request  0.7s
  1.1s          claude_code.llm_request  0.7s
  1.7s          claude_code.tool  0.0s  ToolSearch
  1.7s              claude_code.tool.blocked_on_user  0.0s
  1.7s              claude_code.tool.execution  0.0s
  1.8s          claude_code.llm_request  2.0s
  3.8s          claude_code.tool  4.0s  mcp__site__check_pages
  3.8s              claude_code.tool.blocked_on_user  0.0s
  3.8s              claude_code.tool.execution  4.0s
  7.8s          claude_code.llm_request  0.6s
  8.4s          claude_code.llm_request  1.0s
  3.8s      tools/call check_pages  4.0s
  3.8s          probe /  1.0s
  4.8s          probe /pricing  1.0s
  5.8s          probe /docs  1.0s
  6.8s          probe /login  1.0s


The tool's time appears twice. `claude_code.tool` is Claude Code waiting for it. `tools/call check_pages` is the same call in your process, where an in-process MCP (Model Context Protocol) server runs the tool, and your `probe` spans nest there. Claude Code makes more model requests than sections 2 to 4 because it runs its own loop, with its own tools. With `SENTRY_DSN` set, Claude Code's spans go to Sentry and don't print here.

Claude Code names its token attributes `input_tokens` and `output_tokens`, not `gen_ai.usage.*`, and its `input_tokens` leaves out cached tokens. A Sentry view built on `gen_ai.usage.*` won't count them.

Export errors under the Agent SDK are silent. To see them, set `CLAUDE_CODE_OTEL_DIAG_STDERR=1` and read the `stderr` callback.

## 7. Add Sentry as a Console destination

The spans so far came from your machine. The Claude API can also send its own: one per model request, server tool, and Managed Agents turn. It delivers them straight to Sentry, with no code changes and no exporter in your app.

1. In the Claude Console, open [OpenTelemetry settings](https://platform.claude.com/settings/telemetry-destinations) and choose **Add destination**. You need the admin role.
2. For the provider, choose **Sentry** and paste the DSN. Use the same project as section 1, so the two halves of a trace are in one place. Make sure the destination covers your API key's workspace.
3. Choose a content mode. **Metadata only** sends timing, status, token counts, and the names of your agents and tools. **Full content** adds prompts, responses, and tool input and output.
4. Save, then choose **Test** in the destination's menu and look for the `anthropic.test` span in Sentry.

The destination's content mode doesn't change the SDK's `content_mode`, so set each one.

The SDK sends its trace and span IDs in a `traceparent` header with each request, and the API nests its spans under that. If your tracer provider samples a trace out, the API emits nothing for it. Requests you sent before you saved the destination have no API spans, so run section 2, 3, or 4 again to see them.

## 8. Managed Agents with a custom tool

With Managed Agents, the loop runs on Anthropic's servers. The destination covers the session's turns, model requests, and built-in tools. It can't cover a custom tool, because your app runs that. The SDK's session tool runner runs the tool, sends the result back, and emits `anthropic.custom_tool_use check_pages` from your process. It's async only, so this section uses `AsyncAnthropic`. It reuses `probe_async` from section 6.

Create the agent and its environment once. An environment is the container for the agent's built-in tools. This agent has none, so it gets no outbound network. The tool's name, description, and schema come from the decorated function.

In [ ]:
from anthropic import AsyncAnthropic, beta_async_tool

async_client = AsyncAnthropic()


@beta_async_tool(name="check_pages")
async def check_pages_slow() -> str:
    """Check that every page on the website responds."""
    results = [await probe_async(page) for page in PAGES]
    return json.dumps(results)


agent = await async_client.beta.agents.create(
    name="cookbook-site-health",
    model=MODEL,
    system="You check a website's health. Call check_pages once, then report in one sentence.",
    tools=[
        {
            "type": "custom",
            "name": check_pages_slow.name,
            "description": check_pages_slow.description,
            "input_schema": check_pages_slow.input_schema,
        }
    ],
    metadata={"anthropic_cookbook": "claude-cookbooks/trace-agents-in-sentry"},
)
environment = await async_client.beta.environments.create(
    name="cookbook-site-health",
    config={"type": "cloud", "networking": {"type": "limited"}},
)
print("Agent and environment ready")

Put the whole session under one root span. `initial_events` starts the first turn in the same request that creates the session, so the API's turn span nests under your `sessions.create` span.

`max_idle=2` stops the runner two seconds after the agent ends its turn. Waiting on your tool doesn't count.

In [ ]:
async def health_check(label: str, tool) -> None:
    with tracer.start_as_current_span(f"health_check {label}"):
        session = await async_client.beta.sessions.create(
            agent=agent.id,
            environment_id=environment.id,
            title=f"Site health: {label}",
            initial_events=[
                {"type": "user.message", "content": [{"type": "text", "text": QUESTION}]}
            ],
        )
        try:
            runner = async_client.beta.sessions.events.tool_runner(
                session.id, tools=[tool], max_idle=2
            )
            async for call in runner:
                print("ran", call.name)
            async for event in async_client.beta.sessions.events.list(session.id):
                if event.type in ("agent.message", "session.error"):
                    print(final_text(event) if event.type == "agent.message" else event)
        finally:
            await async_client.beta.sessions.archive(session.id)

    provider.force_flush()  # send the last spans now
    print()
    print_spans()


memory.clear()
await health_check("managed agents", check_pages_slow)

### Find the slow tool call, then fix it

Under `anthropic.custom_tool_use check_pages`, the `probe` spans form a staircase: each page waits for the one before it. You can see it in the start times above. In Sentry, open **Explore > Agents > Traces**, pick the newest `health_check` trace, and look for the longest `anthropic.custom_tool_use` bar.

The checks don't depend on each other, so run them together.

In [ ]:
@beta_async_tool(name="check_pages")
async def check_pages_fast() -> str:
    """Check that every page on the website responds."""
    results = await asyncio.gather(*(probe_async(page) for page in PAGES))
    return json.dumps(results)


await health_check("managed agents, fixed", check_pages_fast)

With the destination from section 7 on, Sentry adds the API's spans to the same trace. The tool result goes back in a second request, and that request starts the agent's second turn. The tree shows one probe and leaves out the event stream and the cleanup requests:

```
health_check managed agents                       your app
├ anthropic.sessions.create                       Anthropic SDK
│ └ anthropic.sessions.create                     Claude API
│   └ anthropic.session.turn                      Claude API
│     └ anthropic.model_request                   Claude API
├ anthropic.custom_tool_use check_pages           Anthropic SDK
│ └ probe /pricing                                your app
└ anthropic.sessions.events.send                  Anthropic SDK
  └ anthropic.sessions.events.send                Claude API
    └ anthropic.session.turn                      Claude API
      └ anthropic.model_request                   Claude API
```

## 9. What Sentry does with the data

| You send | Sentry over OTLP |
|---|---|
| Traces | Accepted. With the Console destination, which Sentry has in open beta, they appear in **Explore > Agents** |
| Logs and log events | Accepted at `/v1/logs` |
| Metrics | [Not supported](https://docs.sentry.io/concepts/otlp/) |
| Span events | [Dropped](https://docs.sentry.io/concepts/otlp/direct/traces/), so Claude Code's `tool.output` content doesn't arrive |
| Array attributes and span links | Shown, not searchable |

- **Count tokens once.** A request can have an SDK span and an API span, and the API's `anthropic.model_request` span under it reports the same usage again. For the Messages API, sum the `anthropic.model_request` spans, and add `anthropic.compaction` if you use compaction. For Managed Agents, sum `anthropic.model_request`, plus `anthropic.session.outcome_evaluation` if you use outcomes.
- **Use one instrumentation.** Sentry's own [Anthropic integration](https://docs.sentry.io/platforms/python/agent-tracing/anthropic/) also traces SDK requests. With both on, each request appears twice. To turn the Anthropic SDK's spans off, pass `open_telemetry=False` to the client.
- **Don't bill from spans.** An unsampled trace or a failing destination drops them. Use the [Usage and Cost API](usage_cost_api.ipynb).

## 10. Clean up

Archiving is permanent. `provider.shutdown()` sends the remaining spans and ends tracing for this kernel.

In [ ]:
await async_client.beta.agents.archive(agent.id)
await async_client.beta.environments.archive(environment.id)
provider.shutdown()
print("Archived and flushed")

## Documentation

- [OpenTelemetry for the Claude API](https://platform.claude.com/docs/en/manage-claude/opentelemetry): [SDK setup](https://platform.claude.com/docs/en/manage-claude/opentelemetry/sdk), [API setup](https://platform.claude.com/docs/en/manage-claude/opentelemetry/api), and [span reference](https://platform.claude.com/docs/en/api/opentelemetry/spans)
- [Claude Code monitoring](https://code.claude.com/docs/en/monitoring-usage) and [Agent SDK observability](https://code.claude.com/docs/en/agent-sdk/observability)
- Sentry: [Claude Platform](https://docs.sentry.io/concepts/otlp/direct/claude-platform/) and [OTLP](https://docs.sentry.io/concepts/otlp/)